In [ ]:
# import dependencies
import pandas as pd
import sqlite3
import warnings

from scipy.stats import chi2_contingency
from scipy.stats import f_oneway

from statsmodels.stats.outliers_influence import variance_inflation_factor


In [140]:
warnings.filterwarnings('ignore')

In [ ]:
# Creating database connection
conn = sqlite3.connect('credit_modelling.db')

In [142]:
# Checking the tables in the database
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)
tables

,name
0,internal_product
1,cibil_score
2,feature_target_description


In [143]:
internal_data = pd.read_sql_query("SELECT * FROM internal_product", conn)
internal_data.head()

,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,...,CC_TL,Consumer_TL,Gold_TL,Home_TL,PL_TL,Secured_TL,Unsecured_TL,Other_TL,Age_Oldest_TL,Age_Newest_TL
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0,0,1,0,4,1,4,0,72,18
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0,1,0,0,0,0,1,0,7,7
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0,6,1,0,0,2,6,0,47,2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0,0,0,0,0,0,1,1,5,5
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0,0,0,0,0,3,0,2,131,32


In [144]:
cibil_data = pd.read_sql_query("SELECT * FROM cibil_score", conn)
cibil_data.head()

,PROSPECTID,time_since_recent_payment,time_since_first_deliquency,time_since_recent_deliquency,num_times_delinquent,max_delinquency_level,max_recent_level_of_deliq,num_deliq_6mts,num_deliq_12mts,num_deliq_6_12mts,...,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,max_unsec_exposure_inPct,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,549,35,15,11,29,29,0,0,0,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,47,-99999,-99999,0,-99999,0,0,0,0,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,302,11,3,9,25,25,1,9,8,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,-99999,-99999,-99999,0,-99999,0,0,0,0,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,583,-99999,-99999,0,-99999,0,0,0,0,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [145]:
internal_data.shape

(51336, 26)

In [146]:
cibil_data.shape

(51336, 62)

In [147]:
internal_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51336 entries, 0 to 51335
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   PROSPECTID            51336 non-null  int64  
 1   Total_TL              51336 non-null  int64  
 2   Tot_Closed_TL         51336 non-null  int64  
 3   Tot_Active_TL         51336 non-null  int64  
 4   Total_TL_opened_L6M   51336 non-null  int64  
 5   Tot_TL_closed_L6M     51336 non-null  int64  
 6   pct_tl_open_L6M       51336 non-null  float64
 7   pct_tl_closed_L6M     51336 non-null  float64
 8   pct_active_tl         51336 non-null  float64
 9   pct_closed_tl         51336 non-null  float64
 10  Total_TL_opened_L12M  51336 non-null  int64  
 11  Tot_TL_closed_L12M    51336 non-null  int64  
 12  pct_tl_open_L12M      51336 non-null  float64
 13  pct_tl_closed_L12M    51336 non-null  float64
 14  Tot_Missed_Pmnt       51336 non-null  int64  
 15  Auto_TL               51336 no

In [148]:
cibil_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51336 entries, 0 to 51335
Data columns (total 62 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   PROSPECTID                    51336 non-null  int64  
 1   time_since_recent_payment     51336 non-null  int64  
 2   time_since_first_deliquency   51336 non-null  int64  
 3   time_since_recent_deliquency  51336 non-null  int64  
 4   num_times_delinquent          51336 non-null  int64  
 5   max_delinquency_level         51336 non-null  int64  
 6   max_recent_level_of_deliq     51336 non-null  int64  
 7   num_deliq_6mts                51336 non-null  int64  
 8   num_deliq_12mts               51336 non-null  int64  
 9   num_deliq_6_12mts             51336 non-null  int64  
 10  max_deliq_6mts                51336 non-null  int64  
 11  max_deliq_12mts               51336 non-null  int64  
 12  num_times_30p_dpd             51336 non-null  int64  
 13  num_times_60

- For internal dataframe, in Age_Oldest_TL column -99999 as a value means that it is a null value in that row.
- Similarly in the cibil dataframe we see for multiple columns with null values. 

In [149]:
def drop_columns_and_rows_with_high_missing_values(df, threshold=10000, null_value=-99999):
    columns_to_drop = []
    df_dropped = df.copy()
    
    for column in df.columns:
        if df.loc[df[column] == null_value].shape[0] > threshold:
            columns_to_drop.append(column)

    # Drop the identified columns
    if columns_to_drop:
        print(f"Dropping columns: {columns_to_drop}")
        df_dropped = df.drop(columns=columns_to_drop)
    
    for column in df_dropped.columns:
        df_dropped = df_dropped.loc[df_dropped[column] != null_value]
    
    return df_dropped


In [150]:
internal_data = drop_columns_and_rows_with_high_missing_values(internal_data)
cibil_data = drop_columns_and_rows_with_high_missing_values(cibil_data)

Dropping columns: ['time_since_first_deliquency', 'time_since_recent_deliquency', 'max_delinquency_level', 'max_deliq_6mts', 'max_deliq_12mts', 'CC_utilization', 'PL_utilization', 'max_unsec_exposure_inPct']


In [151]:
cibil_data.shape

(42066, 54)

In [152]:
internal_data.isna().sum()

PROSPECTID              0
Total_TL                0
Tot_Closed_TL           0
Tot_Active_TL           0
Total_TL_opened_L6M     0
Tot_TL_closed_L6M       0
pct_tl_open_L6M         0
pct_tl_closed_L6M       0
pct_active_tl           0
pct_closed_tl           0
Total_TL_opened_L12M    0
Tot_TL_closed_L12M      0
pct_tl_open_L12M        0
pct_tl_closed_L12M      0
Tot_Missed_Pmnt         0
Auto_TL                 0
CC_TL                   0
Consumer_TL             0
Gold_TL                 0
Home_TL                 0
PL_TL                   0
Secured_TL              0
Unsecured_TL            0
Other_TL                0
Age_Oldest_TL           0
Age_Newest_TL           0
dtype: int64

In [153]:
cibil_data.isna().sum()

PROSPECTID                    0
time_since_recent_payment     0
num_times_delinquent          0
max_recent_level_of_deliq     0
num_deliq_6mts                0
num_deliq_12mts               0
num_deliq_6_12mts             0
num_times_30p_dpd             0
num_times_60p_dpd             0
num_std                       0
num_std_6mts                  0
num_std_12mts                 0
num_sub                       0
num_sub_6mts                  0
num_sub_12mts                 0
num_dbt                       0
num_dbt_6mts                  0
num_dbt_12mts                 0
num_lss                       0
num_lss_6mts                  0
num_lss_12mts                 0
recent_level_of_deliq         0
tot_enq                       0
CC_enq                        0
CC_enq_L6m                    0
CC_enq_L12m                   0
PL_enq                        0
PL_enq_L6m                    0
PL_enq_L12m                   0
time_since_recent_enq         0
enq_L12m                      0
enq_L6m 

Merging both the dataframes using inner join over PROSPECT_ID

In [154]:
combined_dataframe = pd.merge(internal_data, cibil_data, how='inner', left_on=['PROSPECTID'], right_on=['PROSPECTID'])

In [155]:
combined_dataframe.shape

(42064, 79)

In [156]:
combined_dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 79 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PROSPECTID                  42064 non-null  int64  
 1   Total_TL                    42064 non-null  int64  
 2   Tot_Closed_TL               42064 non-null  int64  
 3   Tot_Active_TL               42064 non-null  int64  
 4   Total_TL_opened_L6M         42064 non-null  int64  
 5   Tot_TL_closed_L6M           42064 non-null  int64  
 6   pct_tl_open_L6M             42064 non-null  float64
 7   pct_tl_closed_L6M           42064 non-null  float64
 8   pct_active_tl               42064 non-null  float64
 9   pct_closed_tl               42064 non-null  float64
 10  Total_TL_opened_L12M        42064 non-null  int64  
 11  Tot_TL_closed_L12M          42064 non-null  int64  
 12  pct_tl_open_L12M            42064 non-null  float64
 13  pct_tl_closed_L12M          42064 non-null

We will divide the features into --
- Categorical
- Numerical

Treat them separately

In [157]:
def drop_categorical_by_chi2(df, target_column, p_value_threshold=0.05):
    """
    Performs a Chi-square hypothesis test between all string-type categorical 
    columns and a target column. Drops columns where the p-value is below the threshold.
    """
    columns_to_drop = []
    
    # Identify categorical columns (pandas usually stores strings as 'object' or 'string')
    # We exclude the target column so it doesn't test against itself
    categorical_cols = [
        col for col in df.columns 
        if col != target_column and df[col].dtype in ['str']
    ]

    print(f"Categorical columns to test: {categorical_cols}")
    
    for col in categorical_cols:
        # Create the contingency table
        contingency_table = pd.crosstab(df[col], df[target_column])
        
        # Run the test
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        
        print(f"Tested '{col}': p-value = {p_value}")
        
        # Check against threshold
        if p_value > p_value_threshold:
            columns_to_drop.append(col)
            print(f" -> Dropping '{col}' (p-value > {p_value_threshold})")
            
    # Drop the flagged columns
    df_dropped = df.drop(columns=columns_to_drop)
    
    return df_dropped

In [158]:
combined_dataframe = drop_categorical_by_chi2(combined_dataframe, target_column='Approved_Flag')

Categorical columns to test: ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']
Tested 'MARITALSTATUS': p-value = 3.5781808610388605e-233
Tested 'EDUCATION': p-value = 2.6942265249737532e-30
Tested 'GENDER': p-value = 1.9079361001865664e-05
Tested 'last_prod_enq2': p-value = 0.0
Tested 'first_prod_enq2': p-value = 7.849976105554191e-287


In [159]:
combined_dataframe['MARITALSTATUS'].value_counts()

MARITALSTATUS
Married    30886
Single     11178
Name: count, dtype: int64

In [160]:
combined_dataframe['first_prod_enq2'].value_counts()

first_prod_enq2
others          20640
ConsumerLoan    11075
PL               4431
AL               2641
CC               1988
HL               1289
Name: count, dtype: int64

In [161]:
combined_dataframe['last_prod_enq2'].value_counts()

last_prod_enq2
ConsumerLoan    16480
others          13653
PL               7553
CC               2195
AL               1353
HL                830
Name: count, dtype: int64

In [162]:
combined_dataframe['GENDER'].value_counts()

GENDER
M    37345
F     4719
Name: count, dtype: int64

In [163]:
combined_dataframe['EDUCATION'].value_counts()

EDUCATION
GRADUATE          14140
12TH              11703
SSC                7241
UNDER GRADUATE     4572
OTHERS             2291
POST-GRADUATE      1898
PROFESSIONAL        219
Name: count, dtype: int64

## Hypothesis Tesing 
## Inferential Testing

### Are two columns associated : 

- H0 : Null Hypothesis : Not associated

- H1 : Alternate Hypothesis : Associated

- Alpha(Assumed)
    - Significance level
    - Strictness level
    - Margin level

- Less risky projects = high alpha
- High risky projects = low alpha

- Confidence Interval :
    1 - alpha

- Calculate the evidence against H0
    - p-value
    - Calculated against tests
    - T-test, Chi-square, Anova
    - Degree of freedom

- If p-value <= alpha :
    - Reject H0

- p-value > alpha : 
    -  Fail to reject H0
    - We do not write it as accepting H0 because there is still some chance of wrong outcomes. 


Above mentioned tests and when to use them : 

- Chisquare : Cat vs Cat
- T-test : Cat vs Num (when we have only 2 categories in a column)
- Anova : Cat vs Num (when we have more than 2 categories in a column)

Since all the categorial features have p-val <=0.05, we will accept all.

In [164]:
def filter_numerical_features(df, target_column='Approved_Flag', vif_threshold=6.0, anova_threshold=0.05):
    """
    Evaluates numerical columns by sequentially removing those with high multicollinearity (VIF > threshold),
    then testing the remaining features against a categorical target using ANOVA.
    """
    # 1. Identify numeric columns (excluding the target if it happens to be numeric)
    numeric_columns = [
        col for col in df.columns 
        if df[col].dtype in ['int64', 'float64']
    ]
    
    # 2. VIF Sequential Check
    print("--- Starting VIF Check ---")
    vif_data = df[numeric_columns].copy()
    columns_after_vif = []
    column_index = 0
    
    for col in numeric_columns:
        # Calculate VIF for the column currently at 'column_index'
        vif_value = variance_inflation_factor(vif_data.values, column_index)
        
        if vif_value <= vif_threshold:
            columns_after_vif.append(col)
            print(f"Kept: {col} (VIF: {vif_value})")
            column_index += 1
        else:
            print(f"Dropped: {col} (VIF: {vif_value} > {vif_threshold})")
            vif_data = vif_data.drop(columns=[col])
            
    # 3. ANOVA Test
    print("\n--- Starting ANOVA Check ---")
    final_numerical_columns = []
    
    for col in columns_after_vif:
        # Dynamically separate the column into lists based on the target column's classes
        # This replaces the hardcoded zip(a, b) logic for P1, P2, P3, P4
        category_groups = [group.values for name, group in df.groupby(target_column)[col]]
        
        # Unpack the groups into f_oneway
        f_statistic, p_value = f_oneway(*category_groups)
        
        if p_value < anova_threshold:
            final_numerical_columns.append(col)
            print(f"Kept: {col} (p-value: {p_value:.4f})")
        else:
            print(f"Dropped: {col} (p-value: {p_value:.4f} >= {anova_threshold})")
            
    # 4. Drop the rejected columns and return the new DataFrame
    # Find all numeric columns that didn't make the final cut
    rejected_columns = set(numeric_columns) - set(final_numerical_columns)
    
    df_filtered = df.drop(columns=list(rejected_columns))
    print(f"\nTotal columns dropped: {len(rejected_columns)}")
    
    return df_filtered

In [165]:
combined_dataframe = filter_numerical_features(combined_dataframe)

--- Starting VIF Check ---
Kept: PROSPECTID (VIF: 1.001471445375174)
Dropped: Total_TL (VIF: inf > 6.0)
Dropped: Tot_Closed_TL (VIF: inf > 6.0)
Dropped: Tot_Active_TL (VIF: 11.320645246142803 > 6.0)
Dropped: Total_TL_opened_L6M (VIF: 8.364064272336133 > 6.0)
Dropped: Tot_TL_closed_L6M (VIF: 6.520792569522153 > 6.0)
Kept: pct_tl_open_L6M (VIF: 5.1498060557107745)
Kept: pct_tl_closed_L6M (VIF: 2.611207377167495)
Dropped: pct_active_tl (VIF: inf > 6.0)
Dropped: pct_closed_tl (VIF: 1789.5578922792413 > 6.0)
Dropped: Total_TL_opened_L12M (VIF: 8.601096683974646 > 6.0)
Kept: Tot_TL_closed_L12M (VIF: 3.832851362174614)
Dropped: pct_tl_open_L12M (VIF: 6.099768198336679 > 6.0)
Kept: pct_tl_closed_L12M (VIF: 5.5814467511930905)
Kept: Tot_Missed_Pmnt (VIF: 1.9856551389669772)
Dropped: Auto_TL (VIF: inf > 6.0)
Kept: CC_TL (VIF: 4.809807520859604)
Dropped: Consumer_TL (VIF: 23.27288184101288 > 6.0)
Dropped: Gold_TL (VIF: 30.596519319624697 > 6.0)
Kept: Home_TL (VIF: 4.384812209111584)
Kept: PL_TL (

In [166]:
len(combined_dataframe.columns)

44

## Multicollinearity vs Correlation

- Multicollinearity is basically predictability of each features by other features. Beacuse of multi-collinearity it becomes difficult to reward/penalize the components.
- Correlation is specific to linear relationship between columns. In convex function correlation gives misleading values.


## VIF : Variance Inflation Factor

- Used to identify multicollinearity among IVs
- Takes R-squared value for each IV and eliminates if crosses a threshold.

- VIF_i : 1 / (1 - (R_i)^2)

- VIF ranges from 1 to infinity
- VIF = 1 i.e no multicollinearity
- VIF btw 1 and 5 i.e low multicollinearity
- VIF btw 5 and 10 i.e moderate multicollinearity
- VIF above 10 i.e High multicollinearity

# VIF types :

1. Parallel : considered as a wrong method to use because 2 associated feature can drop each other.
    - Ex: v3, v4, v5 = Multicollinear(VIF very high) out of v1, v2, v3, v4, v5, v6, v7, v8, v9, v10
    - In threshold check remove them v1, v2, v3/v4/v5(anyone), v6, v7, v8, v9, v10
2. Sequential

Check Anova for columns_to_be_kept

## Chi-square Test (Cat vs Cat)
* if pval <= alpha
    - H0 reject
    - H1 accept
    - col A and col B are associated

* if pval > aplha
    - H0 fail to reject(we write like this because we don't have enough proof for it to prove)
    - col A and col B are not associated

## ANOVA (Cat vs Cat/NUM)

* if pvalue <= aplha
    - H0 rject 
    - H1 accept 
    - col A and col B are associated

* if pvalue > alpha
    - Fail to reject H0
    - col A and col B are not associated

## Label Encoding for Categorical Features

In [167]:
combined_dataframe['MARITALSTATUS'].unique()

<StringArray>
['Married', 'Single']
Length: 2, dtype: str

In [168]:
combined_dataframe['EDUCATION'].unique()

<StringArray>
[          '12TH',       'GRADUATE',            'SSC',  'POST-GRADUATE',
 'UNDER GRADUATE',         'OTHERS',   'PROFESSIONAL']
Length: 7, dtype: str

In [169]:
combined_dataframe['GENDER'].unique()

<StringArray>
['M', 'F']
Length: 2, dtype: str

In [170]:
combined_dataframe['last_prod_enq2'].unique()

<StringArray>
['PL', 'ConsumerLoan', 'AL', 'CC', 'others', 'HL']
Length: 6, dtype: str

In [171]:
combined_dataframe['first_prod_enq2'].unique()

<StringArray>
['PL', 'ConsumerLoan', 'others', 'AL', 'HL', 'CC']
Length: 6, dtype: str

### Ordinal feature -- Education
- SSC            : 1
- 12th           : 2
- GRADUATE       : 3
- UNDER GRADUATE : 3
- POST-GRADUATE  : 4
- OTHERS         : 1
- PROFESSIONAL   : 3

Others has to be verified by the businees end user

In [172]:
combined_dataframe.loc[combined_dataframe['EDUCATION']=='SSC', ['EDUCATION']] = '1'
combined_dataframe.loc[combined_dataframe['EDUCATION']=='12TH', ['EDUCATION']] = '2'
combined_dataframe.loc[combined_dataframe['EDUCATION']=='GRADUATE', ['EDUCATION']] = '3'
combined_dataframe.loc[combined_dataframe['EDUCATION']=='UNDER GRADUATE', ['EDUCATION']] = '3'
combined_dataframe.loc[combined_dataframe['EDUCATION']=='POST-GRADUATE', ['EDUCATION']] = '4'
combined_dataframe.loc[combined_dataframe['EDUCATION']=='OTHERS', ['EDUCATION']] = '1'
combined_dataframe.loc[combined_dataframe['EDUCATION']=='PROFESSIONAL', ['EDUCATION']] = '3'

In [173]:
combined_dataframe['EDUCATION'].value_counts()
combined_dataframe['EDUCATION'] = combined_dataframe['EDUCATION'].astype(int)
combined_dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PROSPECTID                 42064 non-null  int64  
 1   pct_tl_open_L6M            42064 non-null  float64
 2   pct_tl_closed_L6M          42064 non-null  float64
 3   Tot_TL_closed_L12M         42064 non-null  int64  
 4   pct_tl_closed_L12M         42064 non-null  float64
 5   Tot_Missed_Pmnt            42064 non-null  int64  
 6   CC_TL                      42064 non-null  int64  
 7   Home_TL                    42064 non-null  int64  
 8   PL_TL                      42064 non-null  int64  
 9   Secured_TL                 42064 non-null  int64  
 10  Unsecured_TL               42064 non-null  int64  
 11  Other_TL                   42064 non-null  int64  
 12  Age_Oldest_TL              42064 non-null  int64  
 13  Age_Newest_TL              42064 non-null  int64  
 14  t

In [174]:
combined_dataframe_encoded = pd.get_dummies(combined_dataframe, columns=['MARITALSTATUS', 'first_prod_enq2', 'last_prod_enq2', 'GENDER'], dtype='uint8')

In [175]:
combined_dataframe_encoded.head()

,PROSPECTID,pct_tl_open_L6M,pct_tl_closed_L6M,Tot_TL_closed_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,CC_TL,Home_TL,PL_TL,Secured_TL,...,first_prod_enq2_PL,first_prod_enq2_others,last_prod_enq2_AL,last_prod_enq2_CC,last_prod_enq2_ConsumerLoan,last_prod_enq2_HL,last_prod_enq2_PL,last_prod_enq2_others,GENDER_F,GENDER_M
0,1,0.000,0.0,0,0.000,0,0,0,4,1,...,1,0,0,0,0,0,1,0,0,1
1,2,0.000,0.0,0,0.000,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2,3,0.125,0.0,0,0.000,1,0,0,0,2,...,0,1,0,0,1,0,0,0,0,1
3,5,0.000,0.0,0,0.000,0,0,0,0,3,...,0,0,1,0,0,0,0,0,0,1
4,6,0.000,0.0,1,0.167,0,0,0,0,6,...,1,0,0,0,1,0,0,0,0,1


In [176]:
combined_dataframe_encoded.shape

(42064, 56)

In [177]:
combined_dataframe_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 56 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   PROSPECTID                    42064 non-null  int64  
 1   pct_tl_open_L6M               42064 non-null  float64
 2   pct_tl_closed_L6M             42064 non-null  float64
 3   Tot_TL_closed_L12M            42064 non-null  int64  
 4   pct_tl_closed_L12M            42064 non-null  float64
 5   Tot_Missed_Pmnt               42064 non-null  int64  
 6   CC_TL                         42064 non-null  int64  
 7   Home_TL                       42064 non-null  int64  
 8   PL_TL                         42064 non-null  int64  
 9   Secured_TL                    42064 non-null  int64  
 10  Unsecured_TL                  42064 non-null  int64  
 11  Other_TL                      42064 non-null  int64  
 12  Age_Oldest_TL                 42064 non-null  int64  
 13  Age_Newest_T

In [179]:
create_table_query = """CREATE TABLE credit_risk_load_data (
    PROSPECTID BIGINT PRIMARY KEY,
    pct_tl_open_L6M FLOAT NOT NULL,
    pct_tl_closed_L6M FLOAT NOT NULL,
    Tot_TL_closed_L12M INT NOT NULL,
    pct_tl_closed_L12M FLOAT NOT NULL,
    Tot_Missed_Pmnt INT NOT NULL,
    CC_TL INT NOT NULL,
    Home_TL INT NOT NULL,
    PL_TL INT NOT NULL,
    Secured_TL INT NOT NULL,
    Unsecured_TL INT NOT NULL,
    Other_TL INT NOT NULL,
    Age_Oldest_TL INT NOT NULL,
    Age_Newest_TL INT NOT NULL,
    time_since_recent_payment INT NOT NULL,
    max_recent_level_of_deliq INT NOT NULL,
    num_deliq_6_12mts INT NOT NULL,
    num_times_60p_dpd INT NOT NULL,
    num_std_12mts INT NOT NULL,
    num_sub INT NOT NULL,
    num_sub_6mts INT NOT NULL,
    num_sub_12mts INT NOT NULL,
    num_dbt INT NOT NULL,
    num_dbt_12mts INT NOT NULL,
    num_lss INT NOT NULL,
    recent_level_of_deliq INT NOT NULL,
    CC_enq_L12m INT NOT NULL,
    PL_enq_L12m INT NOT NULL,
    time_since_recent_enq INT NOT NULL,
    enq_L3m INT NOT NULL,
    EDUCATION INT NOT NULL,
    NETMONTHLYINCOME BIGINT NOT NULL,
    Time_With_Curr_Empr INT NOT NULL,
    CC_Flag INT NOT NULL,
    PL_Flag INT NOT NULL,
    pct_PL_enq_L6m_of_ever FLOAT NOT NULL,
    pct_CC_enq_L6m_of_ever FLOAT NOT NULL,
    HL_Flag INT NOT NULL,
    GL_Flag INT NOT NULL,
    Approved_Flag VARCHAR(50) NOT NULL,
    MARITALSTATUS_Married TINYINT NOT NULL,
    MARITALSTATUS_Single TINYINT NOT NULL,
    first_prod_enq2_AL TINYINT NOT NULL,
    first_prod_enq2_CC TINYINT NOT NULL,
    first_prod_enq2_ConsumerLoan TINYINT NOT NULL,
    first_prod_enq2_HL TINYINT NOT NULL,
    first_prod_enq2_PL TINYINT NOT NULL,
    first_prod_enq2_others TINYINT NOT NULL,
    last_prod_enq2_AL TINYINT NOT NULL,
    last_prod_enq2_CC TINYINT NOT NULL,
    last_prod_enq2_ConsumerLoan TINYINT NOT NULL,
    last_prod_enq2_HL TINYINT NOT NULL,
    last_prod_enq2_PL TINYINT NOT NULL,
    last_prod_enq2_others TINYINT NOT NULL,
    GENDER_F TINYINT NOT NULL,
    GENDER_M TINYINT NOT NULL
);"""

cursor = conn.cursor()
cursor.execute(create_table_query)

In [181]:
pd.read_sql_query("SELECT * FROM credit_risk_load_data", conn)

,PROSPECTID,pct_tl_open_L6M,pct_tl_closed_L6M,Tot_TL_closed_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,CC_TL,Home_TL,PL_TL,Secured_TL,...,first_prod_enq2_PL,first_prod_enq2_others,last_prod_enq2_AL,last_prod_enq2_CC,last_prod_enq2_ConsumerLoan,last_prod_enq2_HL,last_prod_enq2_PL,last_prod_enq2_others,GENDER_F,GENDER_M


In [182]:
combined_dataframe_encoded.to_sql('credit_risk_load_data', conn, if_exists='replace', index=False)

42064

In [183]:
pd.read_sql_query("SELECT * FROM credit_risk_load_data", conn)

,PROSPECTID,pct_tl_open_L6M,pct_tl_closed_L6M,Tot_TL_closed_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,CC_TL,Home_TL,PL_TL,Secured_TL,...,first_prod_enq2_PL,first_prod_enq2_others,last_prod_enq2_AL,last_prod_enq2_CC,last_prod_enq2_ConsumerLoan,last_prod_enq2_HL,last_prod_enq2_PL,last_prod_enq2_others,GENDER_F,GENDER_M
0,1,0.000,0.00,0,0.000,0,0,0,4,1,...,1,0,0,0,0,0,1,0,0,1
1,2,0.000,0.00,0,0.000,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2,3,0.125,0.00,0,0.000,1,0,0,0,2,...,0,1,0,0,1,0,0,0,0,1
3,5,0.000,0.00,0,0.000,0,0,0,0,3,...,0,0,1,0,0,0,0,0,0,1
4,6,0.000,0.00,1,0.167,0,0,0,0,6,...,1,0,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42059,51332,0.333,0.00,0,0.000,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
42060,51333,0.000,0.25,1,0.250,0,0,0,0,2,...,0,1,0,0,0,0,0,1,0,1
42061,51334,0.500,0.50,1,0.500,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,1
42062,51335,0.000,0.00,1,0.500,0,0,0,0,0,...,0,1,0,0,1,0,0,0,1,0
